In [16]:
import scanpy as sc
import pandas as pd

path = "../Data/1.DLPFC/151507/"
adata = sc.read_visium(path, count_file='filtered_feature_bc_matrix.h5',load_images=True)
adata.var_names_make_unique()
sc.pp.highly_variable_genes(adata, flavor="seurat_v3", n_top_genes=3000)# 筛选3000个高变基因
adata = adata[:, adata.var['highly_variable']].copy()
label = pd.read_table(path + "/metadata.tsv")
valid = ~pd.isnull(label['layer_guess_reordered'])# 去空值
adata = adata[valid].copy()
label = label[valid].copy()
sc.pp.normalize_total(adata, target_sum=1e4)# 归一化
sc.pp.log1p(adata)# 对数化
# sc.pp.scale(adata, zero_center=False, max_value=10)
print("adata.shape:", adata.shape)
adata

C:\Users\16562\AppData\Local\Temp\ipykernel_25516\634364256.py:5: FutureWarning: Use `squidpy.read.visium` instead.
  adata = sc.read_visium(path, count_file='filtered_feature_bc_matrix.h5',load_images=True)
e:\Miniconda3\envs\hgm\lib\site-packages\anndata\_core\anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
e:\Miniconda3\envs\hgm\lib\site-packages\anndata\_core\anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


adata.shape: (4221, 3000)


AnnData object with n_obs × n_vars = 4221 × 3000
    obs: 'in_tissue', 'array_row', 'array_col'
    var: 'gene_ids', 'feature_types', 'genome', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'spatial', 'hvg', 'log1p'
    obsm: 'spatial'

In [17]:
type(label['layer_guess_reordered'].values)

numpy.ndarray

In [18]:
import numpy as np
import dhg
from sklearn.neighbors import NearestNeighbors
# 以空间坐标为依据建立超图
# 1) 取空间坐标 (n_spots, 2)
coords = np.asarray(adata.obsm["spatial"], dtype=np.float32)

# 2) KNN: 每个节点找 k 个最近邻（不含自身）
k = 8
nn = NearestNeighbors(n_neighbors=k + 1, metric="euclidean").fit(coords)
indices = nn.kneighbors(coords, return_distance=False)# shape=(n_spots, k + 1)
# 4) 创建 KNN 超图
shg = dhg.Hypergraph(num_v=coords.shape[0], e_list=indices.tolist())
print(f"空间超图构建完成: |V|={shg.num_v}, |E|={shg.num_e}, k={k}")
genes = adata.X.toarray() # (n_spots, n_genes)
nn = NearestNeighbors(n_neighbors=k + 1, metric="correlation").fit(genes)
indices = nn.kneighbors(genes, return_distance=False)# shape=(n_spots, k + 1)
fhg=dhg.Hypergraph(num_v=genes.shape[0], e_list=indices.tolist())
print(f"特征超图构建完成: |V|={fhg.num_v}, |E|={fhg.num_e}, k={k}")

空间超图构建完成: |V|=4221, |E|=4221, k=8
特征超图构建完成: |V|=4221, |E|=4221, k=8


In [19]:
import torch
import torch.nn.functional as F
from model import HGM
    
# ========= 2) 跨视图对比损失（InfoNCE） =========
def infoNCE(p1, p2, temperature=0.2):
    p1=F.normalize(p1, dim=1)
    p2=F.normalize(p2, dim=1)
    logits = torch.mm(p1, p2.t()) / temperature # 相似度矩阵(N, N)
    labels = torch.arange(p1.size(0), device=p1.device)
    # p1->p2 和 p2->p1 对称的对比学习
    loss_12 = F.cross_entropy(logits, labels)
    loss_21 = F.cross_entropy(logits.t(), labels)
    return 0.5 * (loss_12 + loss_21)

In [20]:
# 预处理后的高变基因表达矩阵作为节点特征 (N, 3000)
features = torch.tensor(genes, dtype=torch.float32)
model = HGM(in_dim=features.shape[1], hid_dim=128, out_dim=32, proj_dim=32)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-4)
if torch.cuda.is_available():
    model = model.cuda()
    features = features.cuda()
    shg=shg.to(device='cuda')
    fhg=fhg.to(device='cuda')

epochs = 100
alpha=1.0
beta=0.1
temperature=0.2

from tqdm.auto import tqdm

for epoch in tqdm(range(1,epochs+1)):
    model.train()
    optimizer.zero_grad()
    z, zs, zf, x_hat = model(features, shg, fhg)
    loss_re = F.mse_loss(x_hat, features)# 重建损失
    loss_con=infoNCE(zs, zf, temperature=0.2)# 对比损失
    loss = alpha * loss_re + beta * loss_con
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        tqdm.write(
            f"Epoch {epoch:3d} | recon={loss_re.item():.6f} | contrast={loss_con.item():.6f} | total={loss.item():.6f}"
        )

# 输出卷积后的嵌入结果
model.eval()
with torch.no_grad():
    z, _ ,_,_= model(features, shg, fhg)

print("编码后特征形状:", tuple(z.shape))
print("前3个节点的前5维编码:")
print(z[:3, :5].cpu())

 23%|██▎       | 23/100 [00:06<00:06, 12.20it/s]

Epoch  20 | recon=0.752860 | contrast=7.622688 | total=1.515129


 44%|████▍     | 44/100 [00:07<00:02, 19.57it/s]

Epoch  40 | recon=0.555123 | contrast=7.423641 | total=1.297487


 62%|██████▏   | 62/100 [00:08<00:01, 22.86it/s]

Epoch  60 | recon=0.487343 | contrast=7.246880 | total=1.212031


 83%|████████▎ | 83/100 [00:09<00:00, 22.67it/s]

Epoch  80 | recon=0.466791 | contrast=6.989282 | total=1.165719


100%|██████████| 100/100 [00:10<00:00, 10.00it/s]

Epoch 100 | recon=0.458560 | contrast=6.810954 | total=1.139655
编码后特征形状: (4221, 32)
前3个节点的前5维编码:
tensor([[-0.1571,  0.6180, -0.2842,  1.5334, -0.0081],
        [ 1.1431, -0.7112, -0.1223,  0.2848,  0.2961],
        [-1.6198,  2.5260,  0.0950,  2.2776, -2.1356]])


In [21]:
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, fowlkes_mallows_score

# 1) 取嵌入 z (N, d)
z = z.detach().cpu().numpy()

y_true = pd.Categorical(label['layer_guess_reordered']).codes
true_k = int(np.unique(y_true).size)
print(f"有效样本数：{len(y_true)} | 真实聚类数：{true_k}")
pca = PCA(n_components=20) 
z_eval = pca.fit_transform(z)

有效样本数：4221 | 真实聚类数：7


In [ ]:
def run_clustering_benchmark(adata, z_eval, y_true, true_k, n_neighbors=15, model_name="EEE"):
    """运行 KMeans / mclust / Leiden，并返回分类结果与评估指标。"""
    import rpy2.robjects as ro
    from rpy2.robjects.packages import importr

    # 1) KMeans
    km_labels = KMeans(n_clusters=true_k, random_state=0, n_init=20).fit_predict(z_eval)

    # 2) mclust
    np.random.seed(2020)
    ro.r["set.seed"](2020)
    importr("mclust")
    rmclust = ro.r["Mclust"]

    z_eval_64 = np.asarray(z_eval, dtype=np.float64)
    n_cols = z_eval_64.shape[1]
    r_cols = {f"PC{i+1}": ro.FloatVector(z_eval_64[:, i]) for i in range(n_cols)}
    z_eval_r_df = ro.DataFrame(r_cols)

    res = rmclust(z_eval_r_df, ro.IntVector([int(true_k)]), model_name)
    mclust_labels = np.asarray(res.rx2("classification"), dtype=int) - 1

    # 3) Leiden（按目标簇数搜索最优 resolution）
    adata_eval = adata.copy()
    adata_eval.obsm["X_hgst"] = z_eval
    sc.pp.neighbors(adata_eval, use_rep="X_hgst", n_neighbors=n_neighbors)

    best_diff = 10**9
    best_res = None
    best_labels = None

    for reso in np.linspace(0.1, 4.0, 79):
        sc.tl.leiden(adata_eval, resolution=float(reso), random_state=0, key_added="leiden_tmp")
        cur_labels = adata_eval.obs["leiden_tmp"].to_numpy()
        cur_k = pd.Series(cur_labels).nunique()
        diff = abs(cur_k - true_k)

        if diff < best_diff:
            best_diff = diff
            best_res = float(reso)
            best_labels = cur_labels.copy()

        if diff == 0:
            break

    leiden_labels = pd.Categorical(best_labels).codes

    # 4) 分类结果表
    cluster_df = pd.DataFrame(index=adata.obs_names)
    cluster_df["ground_truth"] = adata.obs["ground_truth"].astype(str).values
    cluster_df["kmeans"] = km_labels
    cluster_df["mclust"] = mclust_labels
    cluster_df["leiden"] = leiden_labels

    # 5) 评估指标
    def eval_scores(y_t, y_p):
        return {
            "ARI": adjusted_rand_score(y_t, y_p),
            "NMI": normalized_mutual_info_score(y_t, y_p),
            "FMI": fowlkes_mallows_score(y_t, y_p),
        }

    results = {
        "KMeans": eval_scores(y_true, km_labels),
        "mclust/GMM": eval_scores(y_true, mclust_labels),
        "Leiden": eval_scores(y_true, leiden_labels),
    }
    res_df = pd.DataFrame(results).T[["ARI", "NMI", "FMI"]]

    return cluster_df, res_df


cluster_df, res_df = run_clustering_benchmark(adata, z_eval, y_true, true_k)

print("\n分类结果（前5行）:")
print(cluster_df.head(5))

print("\n聚类方法评估结果:")
print(res_df.round(4))

# 如需保存分类结果：
# cluster_df.to_csv(path + "/cluster_results_3methods.csv")

fitting ...
  |======================================================================| 100%
Leiden 最优 resolution=0.250, 聚类数=7

分类结果（前10行）:
                   ground_truth  kmeans  mclust  leiden
AAACAACGAATAGTTC-1       Layer1       4       0       0
AAACAAGTATCTCCCA-1       Layer3       2       1       1
AAACAATCTACTAGCA-1       Layer1       1       4       0
AAACACCAATAACTGC-1           WM       5       6       5
AAACAGCTTTCAGAAG-1       Layer6       0       2       4
AAACAGGGTCTATATT-1       Layer6       0       2       5
AAACAGTGTTCCTGGG-1           WM       6       2       5
AAACATTTCCCGGATT-1       Layer5       2       3       2
AAACCACTACACAGAT-1       Layer3       3       1       6
AAACCCGAACGAAATC-1       Layer3       3       1       6

聚类方法评估结果:
               ARI     NMI     FMI
KMeans      0.4606  0.6394  0.5668
mclust/GMM  0.5477  0.6411  0.6474
Leiden      0.4740  0.5984  0.5627
